# Weapon Detection — V2.1 Training

Cleaned version of the original V2.1 Colab notebook.

The original notebook contained repeated environment restoration,
dataset rebuilding, ZIP inspection, and training-resume cells from
the development process. Those repetitions have been consolidated.

## Canonical V2.1 pipeline

**V1 dataset + unique hard examples → verified V2.1 dataset → V1 `best.pt` → YOLO26m at 1280px → 8-epoch V2.1 run**

### Dataset construction

- V1 training images: **12,199**
- Hard-example pool: **2,851**
- Exact duplicates removed: **1,415**
- Unique hard examples added: **1,436**
- Final V2.1 training images: **13,635**
- Validation images: **2,305**
- Test images: **784**
- Classes: Knife, Machette, Handgun, Longgun, Other

The separate continuation experiment from an intermediate checkpoint is documented at the end but is not part of the canonical V2.1 → V3 pipeline.


In [ ]:
# ============================================================
# 1. ENVIRONMENT AND CONFIGURATION
# ============================================================

from pathlib import Path
import hashlib
import shutil
import zipfile
from collections import Counter

from google.colab import drive
from ultralytics import YOLO

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")

V1_ZIP = DRIVE_ROOT / "Final_Dataset_Split" / "Final_Dataset_Split.zip"

HARD_ZIP = (
    DRIVE_ROOT
    / "Weapon_Detection_Training"
    / "V2"
    / "V2_Hard_Examples_Consolidated.zip"
)

V1_BEST = (
    DRIVE_ROOT
    / "Weapon_Detection_Training"
    / "exp1_yolo26m_640_b24"
    / "weights"
    / "best.pt"
)

WORK_ROOT = Path("/content/weapon_detection_v2_1")

DRIVE_OUTPUT = (
    DRIVE_ROOT
    / "Weapon_Detection_Training"
    / "V2.1"
)

V21_RUN_NAME = "v2_1_yolo26m_1280_e8"

print("V1 ZIP:", V1_ZIP)
print("Hard examples:", HARD_ZIP)
print("V1 checkpoint:", V1_BEST)


In [ ]:
# ============================================================
# 2. VERIFY INPUTS
# ============================================================

required = {
    "V1 dataset ZIP": V1_ZIP,
    "Hard-example ZIP": HARD_ZIP,
    "Frozen V1 best.pt": V1_BEST,
}

for name, path in required.items():
    print(f"{name}: {path}")
    print(f"Exists: {path.exists()}")

missing = [name for name, path in required.items() if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(f"- {name}" for name in missing)
    )

print("\n✓ All required inputs found")


In [ ]:
# ============================================================
# 3. EXTRACT SOURCE DATASETS
# ============================================================

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

WORK_ROOT.mkdir(parents=True)

V1_EXTRACT = WORK_ROOT / "v1"
HARD_EXTRACT = WORK_ROOT / "hard"

V1_EXTRACT.mkdir()
HARD_EXTRACT.mkdir()

with zipfile.ZipFile(V1_ZIP, "r") as archive:
    archive.extractall(V1_EXTRACT)

with zipfile.ZipFile(HARD_ZIP, "r") as archive:
    archive.extractall(HARD_EXTRACT)

V1_DATASET = V1_EXTRACT / "Final_Dataset_Split"
HARD_DATASET = HARD_EXTRACT / "V2_Hard_Examples_Consolidated"

assert V1_DATASET.exists()
assert HARD_DATASET.exists()

print("✓ Source datasets extracted")


In [ ]:
# ============================================================
# 4. BUILD V2.1 DATASET
# ============================================================

V21_ROOT = WORK_ROOT / "V2_Training_Dataset_V2.1"

for split in ["train", "val", "test"]:
    (V21_ROOT / split / "images").mkdir(parents=True)
    (V21_ROOT / split / "labels").mkdir(parents=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def get_images(folder):
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

# Copy V1 training set
v1_train_images = get_images(V1_DATASET / "train" / "images")
assert len(v1_train_images) == 12199

v1_hashes = {}

for image in v1_train_images:
    v1_hashes[sha256_file(image)] = image.name

    shutil.copy2(
        image,
        V21_ROOT / "train" / "images" / f"v1_{image.name}"
    )

    label = V1_DATASET / "train" / "labels" / f"{image.stem}.txt"

    if label.exists():
        shutil.copy2(
            label,
            V21_ROOT / "train" / "labels" / f"v1_{label.name}"
        )

# Add unique hard examples
hard_images = get_images(HARD_DATASET / "images")
assert len(hard_images) == 2851

duplicate_count = 0
added_count = 0

for image in hard_images:
    image_hash = sha256_file(image)

    if image_hash in v1_hashes:
        duplicate_count += 1
        continue

    shutil.copy2(
        image,
        V21_ROOT / "train" / "images" / f"hard_{image.name}"
    )

    label = HARD_DATASET / "labels" / f"{image.stem}.txt"

    if label.exists():
        shutil.copy2(
            label,
            V21_ROOT / "train" / "labels" / f"hard_{label.name}"
        )

    added_count += 1

assert duplicate_count == 1415
assert added_count == 1436

print("V1 train:", len(v1_train_images))
print("Hard pool:", len(hard_images))
print("Duplicates skipped:", duplicate_count)
print("Unique hard examples:", added_count)


In [ ]:
# ============================================================
# 5. PRESERVE V1 VALIDATION AND TEST SETS
# ============================================================

def copy_original_split(split):
    source_images = V1_DATASET / split / "images"
    source_labels = V1_DATASET / split / "labels"

    destination_images = V21_ROOT / split / "images"
    destination_labels = V21_ROOT / split / "labels"

    images = get_images(source_images)

    for image in images:
        shutil.copy2(image, destination_images / image.name)

        label = source_labels / f"{image.stem}.txt"

        if label.exists():
            shutil.copy2(label, destination_labels / label.name)

    return len(images)

val_count = copy_original_split("val")
test_count = copy_original_split("test")

assert val_count == 2305
assert test_count == 784

print("Validation:", val_count)
print("Test:", test_count)
print("✓ V1 validation/test preserved unchanged")


In [ ]:
# ============================================================
# 6. CREATE DATASET YAML
# ============================================================

DATA_YAML = V21_ROOT / "dataset.yaml"

DATA_YAML.write_text(
    f"""path: {V21_ROOT.as_posix()}

train: train/images
val: val/images
test: test/images

names:
  0: Knife
  1: Machette
  2: Handgun
  3: Longgun
  4: Other
""",
    encoding="utf-8",
)

print(DATA_YAML.read_text())


In [ ]:
# ============================================================
# 7. FINAL DATASET VERIFICATION
# ============================================================

EXPECTED_IMAGES = {
    "train": 13635,
    "val": 2305,
    "test": 784,
}

for split, expected in EXPECTED_IMAGES.items():
    actual = len(get_images(V21_ROOT / split / "images"))
    print(f"{split}: {actual:,} / {expected:,}")

    if actual != expected:
        raise RuntimeError(
            f"{split} verification failed"
        )

print("\n✓ V2.1 dataset verified")


In [ ]:
# ============================================================
# 8. CANONICAL V2.1 TRAINING
# ============================================================
# Frozen V1 best.pt
#        ↓
# V2.1 dataset
#        ↓
# YOLO26m | 1280px | 8 epochs
# ============================================================

model = YOLO(str(V1_BEST))

results = model.train(
    data=str(DATA_YAML),
    imgsz=1280,
    epochs=8,
    batch=4,
    workers=2,
    device=0,
    seed=42,
    project=str(DRIVE_OUTPUT),
    name=V21_RUN_NAME,
    save=True,
    save_period=-1,
    resume=False,
    val=True,
)

RUN_DIR = DRIVE_OUTPUT / V21_RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print("Best:", BEST_PT)
print("Last:", LAST_PT)


## Optional continuation experiment

The original notebook also contained a separate continuation experiment from an intermediate V2.1 checkpoint. It is preserved conceptually for reproducibility, but it is **not required for the canonical V2.1 → V3 pipeline**.

Use the canonical E8 checkpoint above when following the project pipeline into V3.


In [ ]:
# ============================================================
# OPTIONAL — V2.1 CONTINUATION EXPERIMENT
# ============================================================
# Uncomment only if reproducing the separate continuation run.

# START_PT = LAST_PT
# CONTINUATION_NAME = "v2_1_yolo26m_1280_cont8"
#
# continuation_model = YOLO(str(START_PT))
#
# continuation_model.train(
#     data=str(DATA_YAML),
#     epochs=8,
#     imgsz=1280,
#     batch=4,
#     workers=2,
#     device=0,
#     seed=42,
#     project=str(DRIVE_OUTPUT),
#     name=CONTINUATION_NAME,
#     resume=False,
#     save=True,
#     save_period=-1,
#     plots=False,
# )
